# RAG Step 1: Document Loading and Chunking

Load source text, split it into smaller overlapping passages, and retain source/chunk IDs. Chunking makes embedding and retrieval focused and helps fit model context limits. This notebook uses sample text; replace it with documents you are permitted to use.


In [ ]:
from pathlib import Path

source_documents = [
    {"id": "rag_intro", "text": "RAG stands for Retrieval-Augmented Generation. It combines a retriever with a language model to answer questions using external knowledge.", "source": "sample/rag_intro.txt"},
    {"id": "vector_db", "text": "A typical RAG workflow loads documents, splits them into chunks, embeds the chunks, stores vectors in a database, and retrieves relevant passages for a question.", "source": "sample/vector_db.txt"},
    {"id": "grounding", "text": "ChromaDB stores embeddings and supports semantic similarity search. Retrieved context can ground an answer and reduce unsupported claims, but retrieval does not guarantee correctness.", "source": "sample/grounding.txt"},
 ]

def chunk_text(text: str, chunk_size: int = 40, overlap: int = 8) -> list[str]:
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Require chunk_size > overlap >= 0")
    words = text.split()
    step = chunk_size - overlap
    return [" ".join(words[start:start + chunk_size]) for start in range(0, len(words), step)]

chunks = []
for document in source_documents:
    for chunk_number, text in enumerate(chunk_text(document["text"]), start=1):
        if text:
            chunks.append({
                "id": f"{document['id']}_chunk_{chunk_number}",
                "text": text,
                "metadata": {"source": document["source"], "document_id": document["id"], "chunk": chunk_number},
            })

print(f"Documents: {len(source_documents)} | Chunks: {len(chunks)}")
for item in chunks[:3]:
    print(item["id"], item["metadata"], "->", item["text"])


## Checkpoint

Each chunk has a stable ID, text, and source metadata. Tune chunk size and overlap using representative questions. This simple word-based splitter is for learning; production systems should consider sentence boundaries and the embedding model's tokenizer. Continue with Step 2 to embed and store these chunks.